# Fundamentos de Bases de Datos Relacionales
## Sesión 1: Introducción a las bases de datos relacionales

Este notebook contiene todos los ejemplos de código SQL presentados en la clase.

## Instalación de dependencias

Para ejecutar este notebook, necesitaremos sqlite3 (incluido en Python por defecto) y pandas para visualizar los resultados.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Crear conexión a base de datos SQLite en memoria
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("Conexión establecida exitosamente.")

## 1. Creación de tablas base

Primero, crearemos la estructura relacional presentada en la diapositiva 7 (Normalización).

### Tabla de Clientes

In [ ]:
# Crear tabla clientes
cursor.execute('''
    CREATE TABLE clientes (
        id_cliente INTEGER PRIMARY KEY,
        nombre VARCHAR(100),
        ciudad VARCHAR(50),
        correo VARCHAR(100)
    )
''')

print("Tabla 'clientes' creada exitosamente.")

### Tabla de Productos

In [ ]:
# Crear tabla productos
cursor.execute('''
    CREATE TABLE productos (
        id_producto INTEGER PRIMARY KEY,
        nombre_producto VARCHAR(100),
        categoria VARCHAR(50),
        precio DECIMAL(10, 2)
    )
''')

print("Tabla 'productos' creada exitosamente.")

### Tabla de Sucursales

In [ ]:
# Crear tabla sucursales
cursor.execute('''
    CREATE TABLE sucursales (
        id_sucursal INTEGER PRIMARY KEY,
        nombre_sucursal VARCHAR(100),
        ciudad VARCHAR(50)
    )
''')

print("Tabla 'sucursales' creada exitosamente.")

### Tabla de Ventas (tabla de hechos que relaciona todas las demás)

In [ ]:
# Crear tabla ventas
cursor.execute('''
    CREATE TABLE ventas (
        id_venta INTEGER PRIMARY KEY,
        id_cliente INTEGER,
        id_producto INTEGER,
        id_sucursal INTEGER,
        fecha DATE,
        cantidad INTEGER,
        total DECIMAL(10, 2),
        FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente),
        FOREIGN KEY (id_producto) REFERENCES productos(id_producto),
        FOREIGN KEY (id_sucursal) REFERENCES sucursales(id_sucursal)
    )
''')

print("Tabla 'ventas' creada exitosamente.")

## 2. Inserción de datos de ejemplo

Ahora insertaremos datos consistentes con los ejemplos de la presentación.

In [ ]:
# Insertar clientes
clientes_data = [
    (1, 'María González', 'Valparaíso', 'maria@email.com'),
    (2, 'Carlos Muñoz', 'Santiago', 'carlos@email.com'),
    (3, 'Ana Silva', 'Valparaíso', 'ana@email.com'),
    (4, 'Camila Rojas', 'Valparaíso', 'camila@email.com'),
    (5, 'Luis Pérez', 'Santiago', 'luis@email.com'),
    (6, 'Roberto López', 'Concepción', 'roberto@email.com'),
    (7, 'Fernanda Costa', 'Santiago', 'fernanda@email.com'),
    (8, 'Diego Martínez', 'Valparaíso', 'diego@email.com')
]

cursor.executemany('INSERT INTO clientes VALUES (?, ?, ?, ?)', clientes_data)
print(f"Se insertaron {cursor.rowcount} clientes.")

In [ ]:
# Insertar productos
productos_data = [
    (1, 'Notebook Lenovo', 'Electrónica', 800.00),
    (2, 'Mouse Logitech', 'Accesorios', 25.00),
    (3, 'Teclado Mecánico', 'Accesorios', 120.00),
    (4, 'Monitor LG 24"', 'Electrónica', 250.00),
    (5, 'Webcam HD', 'Accesorios', 60.00),
    (6, 'Auriculares Sony', 'Accesorios', 150.00),
    (7, 'SSD 500GB', 'Almacenamiento', 80.00),
    (8, 'Cable HDMI', 'Accesorios', 15.00)
]

cursor.executemany('INSERT INTO productos VALUES (?, ?, ?, ?)', productos_data)
print(f"Se insertaron {cursor.rowcount} productos.")

In [ ]:
# Insertar sucursales
sucursales_data = [
    (1, 'Tienda Centro Santiago', 'Santiago'),
    (2, 'Tienda Mall Valparaíso', 'Valparaíso'),
    (3, 'Tienda Concepción', 'Concepción'),
    (4, 'E-commerce', 'Virtual')
]

cursor.executemany('INSERT INTO sucursales VALUES (?, ?, ?)', sucursales_data)
print(f"Se insertaron {cursor.rowcount} sucursales.")

In [ ]:
# Insertar ventas con fechas entre junio y julio 2024
ventas_data = [
    (1, 1, 1, 1, '2024-06-05', 1, 800.00),
    (2, 2, 2, 1, '2024-06-08', 3, 75.00),
    (3, 3, 3, 2, '2024-06-12', 1, 120.00),
    (4, 4, 4, 2, '2024-06-15', 1, 250.00),
    (5, 1, 5, 1, '2024-06-18', 2, 120.00),
    (6, 5, 6, 1, '2024-06-20', 1, 150.00),
    (7, 3, 1, 2, '2024-06-22', 1, 800.00),
    (8, 2, 7, 1, '2024-06-25', 2, 160.00),
    (9, 6, 2, 3, '2024-06-28', 5, 125.00),
    (10, 7, 3, 4, '2024-07-01', 1, 120.00),
    (11, 8, 4, 2, '2024-07-03', 1, 250.00),
    (12, 4, 6, 1, '2024-07-05', 2, 300.00),
    (13, 1, 8, 4, '2024-07-08', 10, 150.00),
    (14, 5, 1, 1, '2024-07-10', 1, 800.00),
    (15, 3, 5, 3, '2024-07-12', 3, 180.00)
]

cursor.executemany('INSERT INTO ventas VALUES (?, ?, ?, ?, ?, ?, ?)', ventas_data)
print(f"Se insertaron {cursor.rowcount} registros de ventas.")

# Confirmar los cambios
conn.commit()

## 3. Ejemplos de código SQL del slide 7 - Normalización

### Ejemplo: Obtener nombres de clientes que realizaron órdenes en junio

In [ ]:
# Slide 7: Ejemplo de normalización
sql_query = """
SELECT nombre 
FROM clientes c 
JOIN ventas v ON c.id_cliente = v.id_cliente
WHERE v.fecha BETWEEN '2024-07-01' AND '2024-07-31'
GROUP BY c.id_cliente, nombre
"""

df = pd.read_sql_query(sql_query, conn)
print("Clientes con ventas en julio:")
print(df)

## 4. Ejemplos de código SQL del slide 16 - Clasificación de comandos SQL

### Ejemplo DQL (Data Query Language): SELECT
Obtener personas que se inscribieron a un curso específico

In [ ]:
# Slide 16: Ejemplo DQL - Consulta básica
# Simulamos una tabla de inscripciones (adaptado al contexto de análisis de datos)
cursor.execute('''
    CREATE TABLE inscripciones (
        id_inscripcion INTEGER PRIMARY KEY,
        nombre_estudiante VARCHAR(100),
        correo VARCHAR(100),
        curso VARCHAR(100)
    )
''')

inscripciones_data = [
    (1, 'Juan García', 'juan@email.com', 'Excel Intermedio'),
    (2, 'María López', 'maria@email.com', 'Excel Intermedio'),
    (3, 'Pedro Rodríguez', 'pedro@email.com', 'Python Avanzado'),
    (4, 'Laura Sánchez', 'laura@email.com', 'Excel Intermedio')
]

cursor.executemany('INSERT INTO inscripciones VALUES (?, ?, ?, ?)', inscripciones_data)
conn.commit()

# Ejecutar la consulta del slide 16
sql_dql = """
SELECT nombre_estudiante, correo 
FROM inscripciones 
WHERE curso = 'Excel Intermedio'
"""

df = pd.read_sql_query(sql_dql, conn)
print("Estudiantes inscritos en 'Excel Intermedio':")
print(df)

## 5. Ejemplos de código SQL del slide 19 - Ventajas de SQL

### Ejemplo: Reporte de ventas por categoría con agregaciones

In [ ]:
# Slide 19: Ejemplo avanzado con agregaciones
sql_aggregation = """
SELECT 
    p.categoria, 
    COUNT(*) AS cantidad_ventas, 
    SUM(v.total) AS total_ventas,
    ROUND(AVG(v.total), 2) AS promedio_venta
FROM ventas v
JOIN productos p ON v.id_producto = p.id_producto
WHERE v.fecha BETWEEN '2024-01-01' AND '2024-06-30'
GROUP BY p.categoria
ORDER BY total_ventas DESC
"""

df = pd.read_sql_query(sql_aggregation, conn)
print("Reporte de ventas por categoría (enero a junio):")
print(df)
print(f"\nTotal general de ventas: ${df['total_ventas'].sum():.2f}")

## 6. Ejemplos de código SQL del slide 20 - Errores comunes

### 6.1 Error: Omisión de condiciones de filtrado (WHERE)
❌ INCORRECTO - Sin WHERE (peligroso en producción)

In [ ]:
# EJEMPLO EDUCATIVO: Mostrar el peligro de no usar WHERE
sql_bad = "SELECT COUNT(*) FROM ventas"  # Sin WHERE - trae TODAS las ventas
df = pd.read_sql_query(sql_bad, conn)
print(f"Total de ventas (sin filtro): {df.iloc[0,0]}")
print("\n⚠️  ADVERTENCIA: Sin WHERE, se podrían eliminar TODOS los registros si usaras DELETE o UPDATE")

✅ CORRECTO - Con WHERE para filtrar específicamente

In [ ]:
# CORRECTO: Con filtro WHERE
sql_good = """
SELECT COUNT(*) as total
FROM ventas 
WHERE fecha BETWEEN '2024-06-01' AND '2024-06-30'
"""
df = pd.read_sql_query(sql_good, conn)
print(f"Total de ventas en junio (con filtro WHERE): {df.iloc[0,0]}")

### 6.2 Error: Abuso del SELECT *

In [ ]:
# ❌ INCORRECTO: SELECT * (trae TODAS las columnas)
sql_bad_select = "SELECT * FROM ventas LIMIT 3"
df = pd.read_sql_query(sql_bad_select, conn)
print("❌ SELECT * (trae todas las columnas - ineficiente):")
print(df)

print(f"\nColumnas traídas: {list(df.columns)}")

In [ ]:
# ✅ CORRECTO: Seleccionar solo columnas necesarias
sql_good_select = """
SELECT 
    id_venta,
    id_cliente,
    total
FROM ventas 
LIMIT 3
"""
df = pd.read_sql_query(sql_good_select, conn)
print("\n✅ SELECT especificado (solo columnas necesarias - eficiente):")
print(df)
print(f"\nColumnas traídas: {list(df.columns)}")

### 6.3 Error: Mal uso de JOIN (LEFT JOIN vs INNER JOIN)

In [ ]:
# ❌ INCORRECTO: INNER JOIN (puede perder datos)
# Primero, creemos un cliente sin ventas para demostrar
cursor.execute("INSERT INTO clientes VALUES (99, 'Cliente Sin Ventas', 'Santiago', 'sinventas@email.com)")
conn.commit()

sql_inner_join = """
SELECT 
    c.nombre,
    COUNT(v.id_venta) as total_ventas
FROM clientes c
INNER JOIN ventas v ON c.id_cliente = v.id_cliente
GROUP BY c.id_cliente, c.nombre
ORDER BY total_ventas DESC
"""

df = pd.read_sql_query(sql_inner_join, conn)
print("❌ INNER JOIN (excluye clientes sin ventas):")
print(f"Registros mostrados: {len(df)}")
print(df)

In [ ]:
# ✅ CORRECTO: LEFT JOIN (mantiene todos los clientes)
sql_left_join = """
SELECT 
    c.nombre,
    COUNT(v.id_venta) as total_ventas
FROM clientes c
LEFT JOIN ventas v ON c.id_cliente = v.id_cliente
GROUP BY c.id_cliente, c.nombre
ORDER BY total_ventas DESC
"""

df = pd.read_sql_query(sql_left_join, conn)
print("\n✅ LEFT JOIN (incluye clientes sin ventas):")
print(f"Registros mostrados: {len(df)}")
print(df)
print("\nNota: 'Cliente Sin Ventas' aparece con 0 ventas (no fue excluido)")

## 7. Consultas SQL del Ejercicio Guiado (Slides 24-28)

### Consulta 1: Obtener clientes de Valparaíso

In [ ]:
# Consulta 1 del ejercicio guiado
sql_consulta1 = """
SELECT 
    nombre, 
    correo
FROM clientes
WHERE ciudad = 'Valparaíso'
ORDER BY nombre
"""

df = pd.read_sql_query(sql_consulta1, conn)
print("Consulta 1: Clientes de Valparaíso")
print(df)
print(f"\nTotal de clientes: {len(df)}")

### Consulta 2: Ventas en junio con nombre del cliente

In [ ]:
# Consulta 2 del ejercicio guiado
sql_consulta2 = """
SELECT 
    c.nombre, 
    v.fecha, 
    v.total
FROM ventas v
JOIN clientes c ON v.id_cliente = c.id_cliente
WHERE v.fecha BETWEEN '2024-06-01' AND '2024-06-30'
ORDER BY v.fecha DESC
"""

df = pd.read_sql_query(sql_consulta2, conn)
print("Consulta 2: Ventas en junio")
print(df)
print(f"\nTotal de ventas en junio: ${df['total'].sum():.2f}")
print(f"Cantidad de transacciones: {len(df)}")

### Consulta 3: Productos vendidos por sucursal

In [ ]:
# Consulta 3 del ejercicio guiado
sql_consulta3 = """
SELECT 
    s.nombre_sucursal, 
    p.nombre_producto, 
    SUM(v.cantidad) AS unidades_vendidas,
    SUM(v.total) AS total_generado
FROM ventas v
JOIN productos p ON v.id_producto = p.id_producto
JOIN sucursales s ON v.id_sucursal = s.id_sucursal
GROUP BY s.id_sucursal, s.nombre_sucursal, p.id_producto, p.nombre_producto
ORDER BY s.nombre_sucursal, unidades_vendidas DESC
"""

df = pd.read_sql_query(sql_consulta3, conn)
print("Consulta 3: Productos vendidos por sucursal")
print(df)

print("\n--- Análisis por sucursal ---")
for sucursal in df['nombre_sucursal'].unique():
    total = df[df['nombre_sucursal'] == sucursal]['total_generado'].sum()
    print(f"{sucursal}: ${total:.2f}")

## 8. Análisis avanzado - Insights comerciales

### Ventas por cliente (RFM Analysis - Recency, Frequency, Monetary)

In [ ]:
sql_rfm = """
SELECT 
    c.nombre,
    c.ciudad,
    COUNT(v.id_venta) as frecuencia_compras,
    SUM(v.total) as valor_total_compras,
    ROUND(AVG(v.total), 2) as ticket_promedio,
    MAX(v.fecha) as ultima_compra
FROM clientes c
LEFT JOIN ventas v ON c.id_cliente = v.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY valor_total_compras DESC
"""

df_rfm = pd.read_sql_query(sql_rfm, conn)
print("Análisis RFM de Clientes:")
print(df_rfm)
print(f"\nCliente más valioso: {df_rfm.loc[df_rfm['valor_total_compras'].idxmax(), 'nombre']} (${df_rfm['valor_total_compras'].max():.2f})")

### Comparativa de canales (Tiendas Físicas vs E-commerce)

In [ ]:
sql_canales = """
SELECT 
    CASE 
        WHEN s.nombre_sucursal LIKE '%E-commerce%' THEN 'E-commerce'
        ELSE 'Tienda Física'
    END AS canal,
    COUNT(v.id_venta) as total_transacciones,
    SUM(v.total) as total_ventas,
    ROUND(AVG(v.total), 2) as ticket_promedio
FROM ventas v
JOIN sucursales s ON v.id_sucursal = s.id_sucursal
GROUP BY canal
"""

df_canales = pd.read_sql_query(sql_canales, conn)
print("Análisis de canales de venta:")
print(df_canales)

## 9. Buenas prácticas demostradas

### Siempre prueba con SELECT antes de DELETE o UPDATE

In [ ]:
# PASO 1: Previsualizamos qué se va a modificar
sql_preview = """
SELECT * FROM ventas 
WHERE id_cliente = 1
"""

df_preview = pd.read_sql_query(sql_preview, conn)
print("Paso 1 - Vista previa de registros a modificar:")
print(df_preview)
print(f"\nTotal de registros que se modificarían: {len(df_preview)}")
print("\n✅ BUENA PRÁCTICA: Siempre previsualizas con SELECT antes de hacer cambios")

### Usar alias para mejorar legibilidad

In [ ]:
# ❌ Sin alias - confuso
sql_sin_alias = """
SELECT clientes.nombre, ventas.total, productos.categoria
FROM clientes, ventas, productos
WHERE clientes.id_cliente = ventas.id_cliente
AND ventas.id_producto = productos.id_producto
LIMIT 3
"""

# ✅ Con alias - claro y legible
sql_con_alias = """
SELECT 
    c.nombre as cliente,
    v.total as monto,
    p.categoria as tipo_producto
FROM clientes c
JOIN ventas v ON c.id_cliente = v.id_cliente
JOIN productos p ON v.id_producto = p.id_producto
LIMIT 3
"""

df_alias = pd.read_sql_query(sql_con_alias, conn)
print("✅ BUENA PRÁCTICA: Usar alias (c, v, p) hace el código más legible")
print(df_alias)

## Conclusión

Este notebook ha cubierto:
1. ✅ Creación de tablas con relaciones (Foreign Keys)
2. ✅ Inserción de datos representativos
3. ✅ Consultas DQL (SELECT con WHERE, JOIN, GROUP BY)
4. ✅ Identificación de errores comunes
5. ✅ Buenas prácticas en SQL
6. ✅ Análisis comercial con datos relacionales

**Próximos pasos:** Practicar con problemas reales, optimizar índices, y explorar consultas más complejas con subconsultas y CTEs (Common Table Expressions).